# <font color='blue'> Binary Sentiment Analysis and Naive Bayes Classification <font>

Naive Bayes Implementation in NLTK for binary sentiment analysis on NFT tweets.

<font color='brown'>  Q. How to comment multiple lines in notebook? <font>
    
<font color='brown'>  A. Select all the lines and press Ctrl+/ (on Windows/Linux) or Cmd+/ (on macOS). <font>

# Steps in this Notebook

1. **Load Data**: Load the dataset of NFT tweets.

2. **Preprocess Data**: Clean the text (remove non-alphabetic characters, lowercase, tokenize, remove stopwords, and stem words).

3. **Label Data**: Assign sentiment labels (pos for positive, neg for negative) based on the afinn sentiment score.

4. **Feature Extraction**: Convert text into a feature set based on the top 3000 most common words.

5. **Train/Test Split**: Split the data into training (75%) and testing (25%) sets.

6. **Train Naïve Bayes Classifier**: Train a Naïve Bayes classifier using the processed data.

7. **Evaluate the Model**: Check its accuracy on the test set.


## <font color='blue'> Uploading csv data <font>


In [8]:
import pandas as pd
import os
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

print (os.getcwd()) 

# nltk.download('stopwords')
# nltk.download('punkt')

# Load the data
mydata = pd.read_csv("240115_binance_1002.csv")
mydata.head(4)

/Users/rajanisingh/Desktop/2026-classroom-04


,username,tweet_text,tweet_raw,date,happy,angry,surprise,sad,fear,afinn,bing,sid,bertweet,bertweet_confidence
0,MantaAnalysis,celebration manta list on binance 2000000 air...,Celebration $MANTA list on #Binance - $2000000...,01/15/2024,0.00,0.00,0.00,0.00,0.00,6,3,0.7430,0,0.760314
1,Amiou16,𝙄𝙣𝙫𝙚𝙨𝙩𝙤𝙧𝙨 𝙖𝙣𝙙 𝘽𝙖𝙘𝙠𝙚𝙙 skyark has a lot of inve...,𝙄𝙣𝙫𝙚𝙨𝙩𝙤𝙧𝙨 𝙖𝙣𝙙 𝘽𝙖𝙘𝙠𝙚𝙙SkyArk has a lot of invest...,01/15/2024,0.33,0.00,0.00,0.33,0.33,4,1,0.4005,1,0.751853
2,Cryptoprime00,dedication to those who wait that deadcat boun...,Dedication to those who wait that deadcat boun...,01/15/2024,0.00,0.00,0.33,0.00,0.67,0,0,0.4404,1,0.755133
3,CryptoLalit151,binance indirectly challenging indian sovereig...,#Binance indirectly Challenging Indian Soverei...,01/15/2024,0.05,0.11,0.21,0.05,0.58,4,1,0.9125,-1,0.708018


## <font color='blue'>   Data Cleaning (deal with missing values) and preprocessing <font>


In [10]:
# Remove rows with NaN values in 'tweet_text' column by using dropna() - Pandas DataFrame method
#  Without any arguments, it removes rows where any of the columns contain NaN values.
new_data = mydata.dropna(subset=['tweet_text'])  #


import re
from nltk.corpus import stopwords
#from nltk.stem import PorterStemmer
from nltk.stem import SnowballStemmer
#nltk.download('punkt')
#nltk.download('stopwords')


# Initialize the Stemmer
sb=SnowballStemmer('english') # or #ps = PorterStemmer()

# Define a function for cleaning and tokenizing text
def data_preprocess(text):
    # Remove non-alphabetic characters and lowercase the text
    new_text = re.sub(r'[^\w\s]', '', text)
    lowercase_text = new_text.lower()
    tokens = word_tokenize(lowercase_text) # Word tokenize text
    
    # Remove stopwords and stem the words
    new_tokens = []
    for w in tokens:
        if w not in stopwords.words('english'):
            stem_tokens = sb.stem(w)   # Apply stemming here
            new_tokens.append(stem_tokens)  # Add the stemmed word to new_tokens
    return new_tokens

# Apply the pre-processing function to the tweet_text column
#new_data['tokens'] = new_data['tweet_text'].apply(data_preprocess)
new_data.loc[:,'tokens'] = new_data['tweet_text'].apply(data_preprocess)

# Preview the data to ensure pre-processing is as expected
#cleaned_data.head(6)
new_data[['tweet_text', 'tokens','afinn']].head()

,tweet_text,tokens,afinn
0,celebration manta list on binance 2000000 air...,"[celebr, manta, list, binanc, 2000000, airdrop...",6
1,𝙄𝙣𝙫𝙚𝙨𝙩𝙤𝙧𝙨 𝙖𝙣𝙙 𝘽𝙖𝙘𝙠𝙚𝙙 skyark has a lot of inve...,"[𝙄𝙣𝙫𝙚𝙨𝙩𝙤𝙧𝙨, 𝙖𝙣𝙙, 𝘽𝙖𝙘𝙠𝙚𝙙, skyark, lot, investor...",4
2,dedication to those who wait that deadcat boun...,"[dedic, wait, deadcat, bounc, sinc, 2, week, b...",0
3,binance indirectly challenging indian sovereig...,"[binanc, indirect, challeng, indian, sovereign...",4
4,today mantanetwork becomes the first zk l2 to ...,"[today, mantanetwork, becom, first, zk, l2, li...",7


## <font color='blue'>   Generate positive and negative class based on the 'afinn' score:  <font>
<font color='brown'>positive (afinn>0) and negative (afinn<=0)  <font>

In [11]:
import numpy as np

new_data['class'] = np.where(new_data['afinn'] > 0, 'pos', 'neg')

# Alternate way using lambda function 
#new_data['class'] = new_data['afinn'].apply(lambda score: 'pos' if score > 0 else 'neg')

# Check the balance of the labels and preview the data
class_balance = new_data['class'].value_counts()
data_preview = new_data[['tweet_text', 'afinn', 'class']].head(9)

#outputs the class distribution and the preview of the data (data_preview).
class_balance,data_preview

(class
 neg    585
 pos    417
 Name: count, dtype: int64,
                                           tweet_text  afinn class
 0  celebration manta list on binance  2000000 air...      6   pos
 1  𝙄𝙣𝙫𝙚𝙨𝙩𝙤𝙧𝙨 𝙖𝙣𝙙 𝘽𝙖𝙘𝙠𝙚𝙙  skyark has a lot of inve...      4   pos
 2  dedication to those who wait that deadcat boun...      0   neg
 3  binance indirectly challenging indian sovereig...      4   pos
 4  today mantanetwork becomes the first zk l2 to ...      7   pos
 5  huge airdrop backed by coinbase  zerion   cost...      4   pos
 6  new call  tia   tiamonds  largest tokenized di...     17   pos
 7  spot bitcoin etf custody vs exchange custody w...     -3   neg
 8  wormhole raised 225m from coinbase with a 25b ...      2   pos)

## <font color='blue'> Feature Extraction (Bag of Words) <font>


In [12]:
# Convert DataFrame rows into a list of tuples --> (tokens, class)
t_c_list = [(t, c) for t, c in zip(new_data['tokens'], new_data['class'])]

#Create a List of All Word tokens
w_list = []
for t in new_data['tokens']:
    w_list.extend(t) # ensures that the words are added individually to the list

# Create a frequency distribution of all words -> counts how many times each word appears in w_list
w_count = nltk.FreqDist(w_list)

# Let's use the top 3000 words as features
top_words = list(w_count.keys())[:3000]  #This retrieves the words (keys) from the frequency distribution

#Define a Feature Extraction Function
#input a doc to create a dictionary (f_dict) that contains the top words as keys and (True or False) as values
def extract_features(doc):
    w_set = set(doc) #convert the input doc into a set of unique words
    f_dict = {}    # initializes an empty dictionary
    for w in top_words:
        # For each word w in top_words, it checks if the word is present in w_set
        # If the word is found, it assigns True to that word in the dictionary, otherwise False
         f_dict[w] = True if w in w_set else False    
    return  f_dict

## <font color='blue'> Split Data into Training and Testing Sets <font>


In [13]:
# Created Feature Sets
f_sets = [(extract_features(t), c) for (t, c) in t_c_list]
#extract_features(t): For each document (represented as a list of tokens t), the extract_features() is called

# Split 75% of the data for training, 25% for testing
m = int(len(f_sets)*0.75)
train_set, test_set = f_sets[:m], f_sets[m:]

#test_set-> contains tuples of 'features' (a dictionary of word presence) and 'actual class' labels (pos or neg)

## <font color='blue'> Naive Bayes Classification <font>

In [17]:
from nltk import NaiveBayesClassifier
from collections import defaultdict


#Train the Naive Bayes Classifier
nb = NaiveBayesClassifier.train(train_set)

# Get the list of actual class from the test set and predicted class from the classifier
predicted_class = []
actual_class=[]
for f, c in test_set:
    predicted_class.append(nb.classify(f)) 
    actual_class.append(c)

# Or predicted_class = [nb.classify(f) for f, c in test_set]
# actual_class = [c for f, c in test_set]


nb.show_most_informative_features(15)
# The "informativeness" of a feature is determined by how much it helps distinguish between the different classes

Most Informative Features
                    like = True              pos : neg    =     16.6 : 1.0
                industri = True              pos : neg    =     15.2 : 1.0
                   thank = True              pos : neg    =     13.3 : 1.0
                   asset = True              pos : neg    =     12.6 : 1.0
                   domin = True              pos : neg    =     11.7 : 1.0
                   great = True              pos : neg    =     10.9 : 1.0
                    free = True              pos : neg    =     10.0 : 1.0
                   happi = True              pos : neg    =     10.0 : 1.0
                     big = True              pos : neg    =      9.6 : 1.0
                  friend = True              pos : neg    =      8.3 : 1.0
                    good = True              pos : neg    =      7.1 : 1.0
                    hope = True              pos : neg    =      7.0 : 1.0
                    shit = True              neg : pos    =      6.9 : 1.0

## <font color='blue'> Model Evaluation <font>

In [18]:
# Evaluate the classifier
import nltk.metrics
from nltk.metrics import precision, recall, f_measure, accuracy
from collections import defaultdict

# refsets: key -> class label (e.g., 'positive'),
# value -> set of indices representing the test instances that truly belong to that class.
# checksets: value -> set of indices of test instances that were predicted to belong to the class.
refsets = defaultdict(set)
checksets = defaultdict(set)

for i, (f, c) in enumerate(test_set):
    refsets[c].add(i)
    observed = nb.classify(f)
    checksets[observed].add(i)

# Accuracy
a=round(accuracy(actual_class, predicted_class),2)
print(f"Accuracy of the Naive Bayes classifier is {a*100}%. \n")

# Precision, Recall, F-Measure for pos classes

precision_pos=round(precision(refsets['pos'], checksets['pos']),2)
recall_pos= round(recall(refsets['pos'], checksets['pos']),2)
f_measure_pos=round(f_measure(refsets['pos'], checksets['pos']),2)

# Precision, Recall, F-Measure for neg classes

precision_neg=round(precision(refsets['neg'], checksets['neg']),2)
recall_neg= round(recall(refsets['neg'], checksets['neg']),2)
f_measure_neg=round(f_measure(refsets['neg'], checksets['neg']),2)

#  Print in table format using Panda library <font>

import pandas as pd

# Create a DataFrame from a dictionary

row_labels = ['Positive Class', 'Negative Class']

table = {
    ' Precision(%)': [precision_pos*100, precision_neg*100],
    ' Recall(%)': [recall_pos*100, recall_neg*100],
    ' F-Measure(%)': [f_measure_pos*100, f_measure_neg*100]
}
df = pd.DataFrame(table, index=row_labels)
print(df)

# # Add border to the above table
# def add_border(s):
#     border = '1px solid black'
#     return f'border: {border}'

# # Apply the border style to the DataFrame
# styled_df = df.style.set_table_styles([{'selector': '','props': [('border', '1px solid black')]}])

# styled_df

Accuracy of the Naive Bayes classifier is 70.0%. 

                Precision(%)  Recall(%)  F-Measure(%)
Positive Class          68.0       33.0          44.0
Negative Class          71.0       91.0          80.0
